# 세 자료 10초 통합 준비·학습
MIMIC + 6종 리듬 + 손목 v3를 공통 형식으로 준비합니다.
**현재 손목 정답 코드는 미확정이며, 6종 사용 범위 확인도 미완료입니다. 준비와 학습 완료는 다릅니다.**
손목은 우선 확보한 006·039 두 사람만 검사합니다. 전체 39명 실험이 아닙니다.
미확정 정답은 -1로 보존하고 학습을 중단합니다. 설정을 True로 바꾸는 것만으로 해결되지 않습니다.
모든 다운로드·전처리·학습은 Colab 런타임에서 실행합니다. 약 2GB 다운로드 공간과 처리 공간이 필요합니다.
참가자별 분할, 고정 임계값 0.5, 자료별 평가를 적용합니다. 현재는 성능 결과가 없습니다.


In [ ]:
from pathlib import Path
import subprocess,sys,json
from datetime import datetime,timezone
REPO=Path('/content/capdyssey_integrated')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-train.txt'),'-r',str(REPO/'requirements-data.txt')],check=True)
print('CODE_VERSION',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-u',str(REPO/'scripts/download_data.py'),'mimic_af'],check=True)
subprocess.run([sys.executable,'-u',str(REPO/'scripts/download_six_for_audit.py')],check=True)
subprocess.run([sys.executable,'-u',str(REPO/'scripts/download_longterm_af.py'),'--subjects','006','039'],check=True)
print('THREE_SOURCE_DOWNLOADS_COMPLETE; wrist subset=2/39')


In [ ]:
BUNDLE=REPO/'data/processed'/('integrated_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
subprocess.run([sys.executable,'-u',str(REPO/'scripts/prepare_af_integrated.py'),
    '--mimic',str(REPO/'data/raw/mimic_af'),'--six',str(REPO/'data/raw/six_rhythms'),
    '--wrist',str(REPO/'data/raw/longterm_af_v3'),'--output',str(BUNDLE)],check=True)
manifest=json.loads((BUNDLE/'manifest.json').read_text())
summary={}
for source in manifest['sources']:
    rows=[s for s in manifest['shards'] if s['source']==source]
    summary[source]={'subjects':len({s['subject'] for s in rows}),'windows':sum(s['n'] for s in rows),
        'label_counts':{str(k):sum(s['label_counts'][str(k)] for s in rows) for k in [-1,0,1]}}
print('PREPARATION_SUMMARY',json.dumps(summary,indent=2))


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from train_af_integrated import preflight
status=json.loads((REPO/'data/af_integration_status.json').read_text())
errors=preflight(manifest,status)
if errors:
    print('TRAINING_NOT_EXECUTED: unresolved requirements')
    print(chr(10).join(errors))
else:
    RUN=REPO/'runs'/('integrated_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
    subprocess.run([sys.executable,'-u',str(REPO/'scripts/train_af_integrated.py'),'--bundle',str(BUNDLE),'--output',str(RUN)],check=True)
    print((RUN/'metrics.json').read_text())
